In [91]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
import ast
import itertools
load_dotenv()
API_KEY = os.getenv("API_KEY")
df1 = pd.read_csv('authors_data.csv')
df2 = pd.read_csv('works_ids.csv')


### Filtering co-authors and converting for individual extraction of author_ids

In [ ]:
work_author_ids = df2["author_ids"]
indexes = df1.index[
    (df1["works_count"] >= 5) &
    (df1["works_count"] <= 5000)
].tolist()

subset = df1.loc[indexes, "author_id"]
existing_author_ids = list(subset)
converted = [ast.literal_eval(x) for x in work_author_ids]
single_list_work_author_ids = [author for sublist in converted for author in sublist]
single_list_work_author_ids = list(set(single_list_work_author_ids))

### Removing original authors, so we parse only for new co-authors

In [ ]:
co_author_ids = set()

for author_id in single_list_work_author_ids:
        if author_id not in existing_author_ids:
            co_author_ids.add(author_id)
        else:
              existing_author_ids.remove(author_id)
        
        
total = len(co_author_ids)

17947

### API calling information on all co-authors

In [89]:
import requests
import pandas as pd
from itertools import islice
from time import sleep

rows4 = []
BASE_URL = "https://api.openalex.org/authors"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

session = requests.Session()

for batch_num, author_batch in enumerate(chunked(co_author_ids, 25), start=1):

    id_filter = "id:" + "|".join(author_batch)

    params = {
        "filter": id_filter,
        "select": "display_name,works_count,id,works_api_url,last_known_institutions,summary_stats",
        "per-page": 200,
        "api_key": API_KEY
    }

    try:
        response = session.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Batch {batch_num} failed: {e}")
        continue

    results = data.get("results", [])
    print(f"Batch {batch_num}: fetched {len(results)} authors")

    for author_data in results:
        h_index = author_data.get("summary_stats", {}).get("h_index")
        institutions = author_data.get("last_known_institutions", [])
        country_code = institutions[0].get("country_code") if institutions else None

        rows4.append({
            "author_id": author_data.get("id"),
            "display_name": author_data.get("display_name"),
            "works_count": author_data.get("works_count"),
            "works_api_url": author_data.get("works_api_url"),
            "h_index": h_index,
            "country_code": country_code
        })

    sleep(0.2)  # small pause to avoid 429

df4 = pd.DataFrame(rows4)

print("Done:", len(df4))

Batch 1: fetched 25 authors
Batch 2: fetched 25 authors
Batch 3: fetched 25 authors
Batch 4: fetched 25 authors
Batch 5: fetched 25 authors
Batch 6: fetched 25 authors
Batch 7: fetched 25 authors
Batch 8: fetched 25 authors
Batch 9: fetched 25 authors
Batch 10: fetched 25 authors
Batch 11: fetched 25 authors
Batch 12: fetched 25 authors
Batch 13: fetched 25 authors
Batch 14: fetched 25 authors
Batch 15: fetched 25 authors
Batch 16: fetched 25 authors
Batch 17: fetched 25 authors
Batch 18: fetched 25 authors
Batch 19: fetched 25 authors
Batch 20: fetched 25 authors
Batch 21: fetched 25 authors
Batch 22: fetched 25 authors
Batch 23: fetched 25 authors
Batch 24: fetched 25 authors
Batch 25: fetched 25 authors
Batch 26: fetched 25 authors
Batch 27: fetched 25 authors
Batch 28: fetched 25 authors
Batch 29: fetched 25 authors
Batch 30: fetched 25 authors
Batch 31: fetched 25 authors
Batch 32: fetched 25 authors
Batch 33: fetched 25 authors
Batch 34: fetched 25 authors
Batch 35: fetched 25 au

In [90]:
df4.to_csv("co_authors_data.csv", index=False)